In [1]:
# ============================================================
# 【第0格】配置路径 - 只需修改这里的 BASE_DIR
# ============================================================
import os

# ✅ 改成你本地的 AMBER-master 根目录（Windows 用 r"G:\...", Linux/Mac 用普通路径）
BASE_DIR = r"G:\sample\Qwen3-VL\AMBER-master"

# 自动拼接各文件路径（无需再改）
PATH_WORD_ASSOCIATION = os.path.join(BASE_DIR, "data", "relation.json")
PATH_INFERENCE_DATA   = os.path.join(BASE_DIR, "qwen3_baseline_output_new.json")
PATH_ANNOTATION = os.path.join(BASE_DIR, "data", "annotations.json")
PATH_SAFE_WORDS       = os.path.join(BASE_DIR, "data", "safe_words.txt")
PATH_METRICS          = os.path.join(BASE_DIR, "data", "metrics.txt")

SIMILARITY_SCORE  = 0.8
EVALUATION_TYPE   = 'g'   # 'g'=生成式  'd'=判别式  'a'=全部

print("路径配置完成：")
for name, p in [("word_association", PATH_WORD_ASSOCIATION),
                ("inference_data",   PATH_INFERENCE_DATA),
                ("annotation",       PATH_ANNOTATION),
                ("safe_words",       PATH_SAFE_WORDS),
                ("metrics",          PATH_METRICS)]:
    print(f"  {name}: {p}")


路径配置完成：
  word_association: G:\sample\Qwen3-VL\AMBER-master\data\relation.json
  inference_data: G:\sample\Qwen3-VL\AMBER-master\qwen3_baseline_output_new.json
  annotation: G:\sample\Qwen3-VL\AMBER-master\data\annotations.json
  safe_words: G:\sample\Qwen3-VL\AMBER-master\data\safe_words.txt
  metrics: G:\sample\Qwen3-VL\AMBER-master\data\metrics.txt


In [2]:
# ============================================================
# 【第1格】导入库 & 工具函数
# ============================================================
import nltk
from nltk.stem import WordNetLemmatizer
import json
import spacy
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
nlp = spacy.load("en_core_web_lg")


def check_synonyms_word(word1, word2, similarity_score):
    token1 = nlp(word1)
    token2 = nlp(word2)
    return token1.similarity(token2) > similarity_score


def extract_nouns(text):
    lemmatizer = WordNetLemmatizer()
    tokens = nltk.word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(w) for w, pos in tagged if pos.startswith('NN')]


def load_metrics(metrics_path):
    """从 metrics.txt 初始化指标字典"""
    metrics = {}
    with open(metrics_path, "r") as f:
        for line in f:
            parts = line.strip().split('=')
            if len(parts) == 2:
                metrics[parts[0].strip()] = eval(parts[1].strip())
    return metrics


def load_inference_data(inference_data_path):
    """
    【修复】兼容两种 JSON 格式：
      - 旧格式（list）：[{"id": 1, "response": "..."}, ...]
      - Qwen3 实际输出格式（dict）：{"1": "...", "2": "...", ...}
    统一转换为 list of dict。
    """
    raw = json.load(open(inference_data_path, 'r', encoding='utf-8'))
    if isinstance(raw, list):
        return raw   # 已经是标准格式
    elif isinstance(raw, dict):
        # key 为字符串 id，value 为 response 字符串
        converted = [
            {"id": int(k), "response": v}
            for k, v in sorted(raw.items(), key=lambda x: int(x[0]))
        ]
        print(f"[load_inference_data] 检测到 dict 格式，已自动转换为 list（共 {len(converted)} 条）")
        return converted
    else:
        raise ValueError(f"无法识别的 inference_data 格式：{type(raw)}")


print("工具函数定义完成 ✓")


工具函数定义完成 ✓


In [3]:
# ============================================================
# 【第2格】主评测逻辑
# ============================================================
def main(word_association_path, inference_data_path, annotation_path,
         safe_words_path, metrics_path, similarity_score, evaluation_type):

    metrics = load_metrics(metrics_path)

    association = json.load(open(word_association_path, 'r', encoding='utf-8'))
    hallucination_words = []
    for word1 in association.keys():
        hallucination_words.append(word1)
        for word2 in association[word1]:
            hallucination_words.append(word2)

    global_safe_words = []
    with open(safe_words_path, 'r', encoding='utf-8') as f:
        for line in f:
            global_safe_words.append(line.split('\n')[0])

    # 确定评测维度
    dimension = {'g': False, 'de': False, 'da': False, 'dr': False}
    if evaluation_type == 'a':
        for key in dimension: dimension[key] = True
    elif evaluation_type == 'g':
        dimension['g'] = True
    elif evaluation_type == 'd':
        dimension['de'] = dimension['da'] = dimension['dr'] = True
    else:
        dimension[evaluation_type] = True

    # 【修复】使用兼容加载函数
    inference_data = load_inference_data(inference_data_path)
    ground_truth   = json.load(open(annotation_path, 'r', encoding='utf-8'))

    for i in tqdm(range(len(inference_data))):
        id_val   = inference_data[i]['id']
        response = inference_data[i]['response']
        gt       = ground_truth[id_val - 1]   # ground_truth 以 0-indexed list 存储

        if gt['type'] == 'generative':
            nouns = extract_nouns(response)
            after_process_nouns = [n for n in nouns if n in hallucination_words]

            safe_words, safe_list = [], []
            for idx, word in enumerate(gt['truth']):
                safe_words += association[word]
                safe_list  += [idx] * len(association[word])

            ha_words, ha_list = [], []
            for idx, word in enumerate(gt['hallu']):
                ha_words += association[word]
                ha_list  += [idx] * len(association[word])

            safe_words += gt['truth']
            safe_len    = len(gt['truth'])
            safe_list  += [0] * safe_len
            safe_flag_list = [0] * len(after_process_nouns)

            ha_words += gt['hallu']
            ha_len    = len(gt['hallu'])
            ha_list  += [0] * ha_len

            for idx, noun in enumerate(after_process_nouns):
                if noun in global_safe_words:
                    continue
                if noun in safe_words:
                    for j in range(len(safe_words)):
                        if noun == safe_words[j]:
                            if j < (len(safe_list) - safe_len):
                                safe_list[safe_list[j] + len(safe_list) - safe_len] = 1
                            else:
                                safe_list[j] = 1
                            break
                    continue
                if noun in ha_words:
                    for j in range(len(ha_words)):
                        if noun == ha_words[j]:
                            if j < (len(ha_list) - ha_len):
                                ha_list[ha_list[j] + len(ha_list) - ha_len] = 1
                            else:
                                ha_list[j] = 1
                            break
                for j, check_word in enumerate(ha_words):
                    if check_synonyms_word(noun, check_word, similarity_score):
                        if j < (len(ha_list) - ha_len):
                            ha_list[ha_list[j] + len(ha_list) - ha_len] = 1
                        else:
                            ha_list[j] = 1
                        break
                flag = False
                for j, check_word in enumerate(safe_words):
                    if check_synonyms_word(noun, check_word, similarity_score):
                        flag = True
                        if j < (len(safe_list) - safe_len):
                            safe_list[safe_list[j] + len(safe_list) - safe_len] = 1
                        else:
                            safe_list[j] = 1
                        break
                if flag:
                    continue
                safe_flag_list[idx] = 1

            metrics['chair_score']       += sum(safe_flag_list)
            metrics['chair_num']          += len(safe_flag_list)
            metrics['safe_cover_score']   += sum(safe_list[-safe_len:])
            metrics['safe_cover_num']     += len(safe_list[-safe_len:])
            metrics['hallu_cover_score']  += sum(ha_list[-ha_len:])
            metrics['hallu_cover_num']    += len(ha_list[-ha_len:])
            if sum(safe_flag_list) == 0:
                metrics['non_hallu_score'] += 1
            metrics['non_hallu_num'] += 1

        else:
            truth = gt['truth']
            q_type = gt['type']

            def add_num(base, sub):
                metrics[base + '_qa_correct_num'] += 1
                metrics[sub  + '_qa_correct_num'] += 1

            metrics['qa_correct_num'] += 1
            if   q_type == 'discriminative-attribute-state':     metrics['as_qa_correct_num'] += 1
            elif q_type == 'discriminative-attribute-number':    metrics['an_qa_correct_num'] += 1
            elif q_type == 'discriminative-attribute-action':    metrics['aa_qa_correct_num'] += 1
            elif q_type == 'discriminative-hallucination':       metrics['ha_qa_correct_num'] += 1
            else:                                                  metrics['asso_qa_correct_num'] += 1

            def bump(base):
                metrics['qa_correct_score'] += 1
                metrics[base + '_qa_correct_score'] += 1

            def bump_no(base):
                metrics['qa_no_score'] += 1
                metrics[base + '_qa_no_score'] += 1
                bump(base)

            sub = ('as' if q_type == 'discriminative-attribute-state' else
                   'an' if q_type == 'discriminative-attribute-number' else
                   'aa' if q_type == 'discriminative-attribute-action' else
                   'ha' if q_type == 'discriminative-hallucination'    else 'asso')

            if truth == 'yes':
                if response == 'Yes':
                    bump(sub)
            else:
                metrics['qa_no_num']          += 1
                metrics[sub + '_qa_no_num']   += 1
                if response == 'No':
                    bump_no(sub)

            if response == 'No':
                metrics['qa_ans_no_num']        += 1
                metrics[sub + '_qa_ans_no_num'] += 1
                if truth == 'no':
                    metrics['qa_ans_no_score']        += 1
                    metrics[sub + '_qa_ans_no_score'] += 1

    # ── 打印结果 ────────────────────────────────────────────────────
    if dimension['g']:
        CHAIR = round(metrics['chair_score']       / metrics['chair_num']       * 100, 1)
        Cover = round(metrics['safe_cover_score']  / metrics['safe_cover_num']  * 100, 1)
        Ha    = round(metrics['hallu_cover_score']  / metrics['hallu_cover_num'] * 100, 1)
        Ha_p  = round(100 - metrics['non_hallu_score'] / metrics['non_hallu_num'] * 100, 1)
        print("===== Generative Task =====")
        print(f"CHAIR : {CHAIR}")
        print(f"Cover : {Cover}")
        print(f"Hal   : {Ha_p}")
        print(f"Cog   : {Ha}")

    if dimension['de'] and dimension['da'] and dimension['dr']:
        Accuracy  = round(metrics['qa_correct_score'] / metrics['qa_correct_num'] * 100, 1)
        Precision = round(metrics['qa_ans_no_score']  / metrics['qa_ans_no_num']  * 100, 1)
        Recall    = round(metrics['qa_no_score']       / metrics['qa_no_num']      * 100, 1)
        F1 = round(2*(Precision/100)*(Recall/100)/((Precision/100)+(Recall/100)+1e-4)*100, 1)
        print("===== Discriminative Task =====")
        print(f"Accuracy  : {Accuracy}")
        print(f"Precision : {Precision}")
        print(f"Recall    : {Recall}")
        print(f"F1        : {F1}")

    return metrics


print("main() 定义完成 ✓")


main() 定义完成 ✓


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
print("NLTK 数据包下载完成 ✓")

In [4]:
# ============================================================
# 【第3格】▶ 运行评测（直接执行此格即可）
# ============================================================
metrics_result = main(
    word_association_path = PATH_WORD_ASSOCIATION,
    inference_data_path   = PATH_INFERENCE_DATA,
    annotation_path       = PATH_ANNOTATION,
    safe_words_path       = PATH_SAFE_WORDS,
    metrics_path          = PATH_METRICS,
    similarity_score      = SIMILARITY_SCORE,
    evaluation_type       = EVALUATION_TYPE,
)


[load_inference_data] 检测到 dict 格式，已自动转换为 list（共 500 条）


100%|██████████| 500/500 [00:12<00:00, 41.30it/s] 

===== Generative Task =====
CHAIR : 3.8
Cover : 39.2
Hal   : 14.6
Cog   : 0.3
